In [2]:
using DifferentialEquations # for the actual time evolution
using OrdinaryDiffEq # for ODEs
using Plots # for plotting
using Base.Threads # for parallelization
using StaticArrays # somehow needed to use multiple variables in DifferentialEquations.jl

using Plots, LaTeXStrings, Colors
using Plots.PlotMeasures
using LinearAlgebra

using Random, Distributions

using FFTW # discrete Fourier transform

using JLD2 # for file saving

In [26]:
level = "../../../../"

include(joinpath(level, "src/4th-order-FD-stencils.jl"));
include(joinpath(level, "src/evolution_Liouville_larger_cutoff.jl"));
include(joinpath(level, "src/hamiltonian_Liouville.jl"));
include(joinpath(level, "src/initial_data_waves.jl"));
include(joinpath(level, "src/visualisation.jl"));

### evolution

In [27]:
function artisan_evolution_at_resolution(Nx, stableRandomSeed, pModel, pInit, target_time)
    # unpack model parameters
    (mphi2, mchi2, c4, c, epsDiss) = pModel;
    
    # set the spatial discretization
    NboundaryPadding = 2;#Int(div(Nx,2));  # Number of boundary padding points
    dx = 1/(Nx);  # Grid spacing
        
    pGrid = (dx, Nx, NboundaryPadding);
    # reset a combined set of parameters (residual from old structure ... could be modified)
    # TODO: modify to pGrid, pModel, pInit
    p = (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding);

    # set the time span
    tspan = (0, target_time);

    # set the evolution method
    time_integration_method = RK4();
    
    # generate initial conditions
    u0 = initial_data(
        range(0, step=dx, length=(Nx + 2 * NboundaryPadding)),
        p, 
        pInit
    );
        
    # set the problem
    prob = ODEProblem(finite_differenced_pde_with_bc!, u0, tspan, p);

    sol = solve(
        prob, time_integration_method, 
        saveat = tspan[end]/10^3, #exp.(range(log(tspan[1]), log(tspan[end]), length=10^4))
        dt=dx/10, 
        adaptive = false, 
        dense=false, 
        maxiters=typemax(Int),
        callback=field_size_callback
    );
        
    # obtain the hamiltonian
    hamiltonian = zeros(length(sol.u))
    hamphi = zeros(length(sol.u))
    hamchi = zeros(length(sol.u))
    for i = 1:length(sol.u)
        hamiltonian[i] = nintegrate_simps(hamiltonian_density(sol.u[i], p), dx)
        hamphi[i] = nintegrate_simps(hamiltonian_phi(sol.u[i], p), dx)
        hamchi[i] = nintegrate_simps(hamiltonian_chi(sol.u[i], p), dx)
    end
    
    return (p, sol, hamiltonian, hamphi, hamchi)
end

artisan_evolution_at_resolution (generic function with 1 method)

In [28]:
function evolution_at_param(param, current_target_time, current_res_log2, stableRandomSeed)
    
    #stableRandomSeed = 42
    print("persistent random seed: ", stableRandomSeed, "\n")
    
    print("current mass: ", param, "\n")
    
    # set monitoring flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # parameters of the model
    mphi2 = param^2.;
    mchi2 = param^2.;
    c4 = 1.0;
    c = -1.;
    epsDiss = 0;

    # parameters of the initial data
    a0phi = 0; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    a0chi = a0phi; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    k0phi = 1;
    k0chi = 2 * k0phi;
    x0phi = 0;
    x0chi = 1/3;

    offsetphi = 0;
    offsetchi = 0;

    aStochastic = 4;
    mink = 1;
    maxk = 4;
    
    desiredTkinPhi = NaN;
    desiredTkinChi = NaN;
    
    # set the combined set of parameters 
    pModel = (mphi2, mchi2, c4, c, epsDiss);
    pInit = (
        a0phi, a0chi, k0phi, k0chi, x0phi, x0chi, 
        offsetphi, offsetchi, 
        aStochastic, mink, maxk, stableRandomSeed,
        desiredTkinPhi, desiredTkinChi
    );
     
    # set some tables to store intermediate output
    resTab = [2^i for i in current_res_log2-2:current_res_log2]
    pTab = []
    solTab = []
    hamiltonianTab = []
    hamPhiTab = []
    hamChiTab = []
    
    #############################
    # evolution
    #############################
    
    # run evolution
    for res in resTab
        print("current resolution: ", res, "\n")
        # run the evolution
        @time (p, sol, hamiltonian, hamPhi, hamChi) = artisan_evolution_at_resolution(
            res, stableRandomSeed, pModel, pInit, current_target_time
        )
        print("... terminated", "\n")
        # unpack parameters
        (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding) = p
        # append the results
        push!(pTab, p)
        push!(solTab, sol)
        push!(hamiltonianTab, hamiltonian)
        push!(hamPhiTab, hamPhi)
        push!(hamChiTab, hamChi)
    end
    
    #############################
    # CONVERGENCE
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",param)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        print("Output plot directory does not exist. Creating it ...\n")
        mkpath(dir_path)
    else
        print("Output plot directory already exists.\n")
    end
    
    # plot and determine convergence 
    loss_of_convergence_time = save_convergence_plots(
        resTab, pTab, solTab, 
        hamiltonianTab, 
        dir_path
    )
    if loss_of_convergence_time >= solTab[end].t[end]
        print("Convergence kept at all times.\n")
    else
        print("Convergence lost at time t=",loss_of_convergence_time,"\n")
    end
    
    # determine the index of convergence loss
    loss_of_convergence_index = findfirst(t -> t > loss_of_convergence_time, solTab[end].t)
    if loss_of_convergence_index === nothing
        loss_of_convergence_index = length(solTab[end].t)
    end
    
    #print(10 * hamPhiTab[end][1],"\n")
    #print(hamPhiTab[end][2:10],"\n")
    
    # determine the onset time of the runaway (10-fold increase in either kinetic energy)
    runaway_index_phi = findfirst(energy -> abs(energy) > 10 * abs(hamPhiTab[end][1]), hamPhiTab[end])
    if runaway_index_phi === nothing
        runaway_index_phi = length(hamPhiTab[end])
    end
    runaway_index_chi = findfirst(energy -> abs(energy) > 10 * abs(hamChiTab[end][1]), hamChiTab[end])
    if runaway_index_chi === nothing
        runaway_index_chi = length(hamChiTab[end])
    end
    runaway_index = min(runaway_index_phi, runaway_index_chi)
    runaway_time = solTab[end].t[runaway_index]
    if runaway_time >= solTab[end].t[end]
        print("No runaway detected.\n")
    else
        print("Runaway detected at time t=",runaway_time,"\n")
    end    
    
    #############################
    # GENERATE REMAINING PLOTS IF DESIRED
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",param)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        #print("Directory does not exist. Creating it...")
        mkpath(dir_path)
    end
        
    # plot energy components
    save_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_normalised_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_difference_in_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    
    # plot field heatmaps
    save_density_plots(
        solTab[end], pTab[end], pInit,
        dir_path,
        loss_of_convergence_time=loss_of_convergence_time
    )
    
    # save snapshots
    save_snaps(
        solTab[end], pTab[end];
        snap_intervals=Int(round(length(solTab[end])/1)), 
        yrangeVal=1.2,
        dir_path = dir_path
    );
    
    # animate the fields
    save_animation(
        solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
        pTab[end],
        join([dir_path, "/animation_Nx=", resTab[end], ".gif"])
    );    
#     # animate frequencies
#     save_animation_momentum_space(
#         solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
#         pTab[end],
#         join([dir_path, "/animation_momentum_space_Nx=", resTab[end], ".gif"])
#     );
    
    print("Finished plotting.", "\n")
    
    #############################
    # SAVE DATA
    #############################
    
    if runaway_time > loss_of_convergence_time
        print("WARNING: Convergence not maintained until onset of runaway. Resolution insufficient.", "\n")
    else
        convergence_maintained = true
        if runaway_time >= solTab[end].t[end]
            print("WARNING: Lower bound only because target time insufficient.", "\n")
        else
            lower_bound_only = false
        end
    end
    
    dir_path = string("dat/",stableRandomSeed)
    if !isdir(dir_path)
        mkpath(dir_path)
    end

    timesteps = solTab[end].t
    stable_until = min(runaway_time, loss_of_convergence_time)

    @save joinpath(pwd(), dir_path, string(param,".jld2")) param stable_until lower_bound_only timesteps hamiltonianTab hamPhiTab hamChiTab

    print("Saved data.", "\n")

    
    return (runaway_time, convergence_maintained, lower_bound_only)
end

evolution_at_param (generic function with 1 method)

### main()

In [29]:
#param_base = 1.2
#param_table = reverse([param_base^i for i in -8:8])

param_table = [mass for mass in 0:32:512]

17-element Vector{Int64}:
   0
  16
  32
  48
  64
  80
  96
 112
 128
 144
 160
 176
 192
 208
 224
 240
 256

In [30]:
function main()
    
    # some random seed (can be modified at will)
    stableRandomSeed = rand(1:10^7)
    
    # initialise flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # set abort criteria ...
    highest_res_log2 = 13;
    max_target_time = 2 * 10^2;
    # ... and their initial values
    current_res_log2 = 11;
    current_target_time = 1;
    
    # initialise the runaway time for handover to next param value
    runaway_time = Inf;
    
    # set table of desired param_table (NOTE: links to scaling assumption below)
    param_base = 1
    param_table = [mass for mass in 0:32:512]
    
    # loop over all values in param_table
    for param in param_table
        
        # re-attempt while flags not positive or until abort criteria met
        while (!convergence_maintained||lower_bound_only) && (current_res_log2 <= highest_res_log2) && (current_target_time <= max_target_time)
            # attempt run and obtain flags
            (runaway_time, convergence_maintained, lower_bound_only) = evolution_at_param(
                param, 
                current_target_time, 
                current_res_log2,
                stableRandomSeed
            )
            # update according to obtained flags
            if convergence_maintained                
                if lower_bound_only
                    current_target_time = current_target_time * 4
                    print("Increasing target time to T = ", current_target_time, "\n")
                else
                    current_target_time = min(runaway_time, current_target_time);
                    print("Target time reset to confidently detected runaway time T = ", current_target_time, "\n")
                end
            else
                current_res_log2 = current_res_log2 + 1;
                print("Increasing resolution from N = ", current_res_log2 - 1, " to ", current_res_log2, "\n")
            end
        end
        
        print("PARAM = ", param, " DONE!\n")
        
        # update target time based on the presumed scaling assumption and adapt the target time accordingly
        current_target_time = runaway_time
        print("Updating target time for next param value from T = ", current_target_time, " ... ")
        current_target_time = current_target_time * exp(param_base)     
        print("to T = ", current_target_time, "\n")
        
        # decrease resolution if convergence was maintained in previous step
#         if convergence_maintained
#             current_res_log2 = current_res_log2 - 1;
#             print("Decreasing resolution from N = ", current_res_log2 + 1, " to ", current_res_log2, "\n")
#         end
        
        # check whether it makes sense to go on; otherwise abort 
        if convergence_maintained && lower_bound_only && current_target_time >= max_target_time
            print("ABORT: maximum target time approached in converged simulation; no use to proceed")
            return
        end
        
        # ensure that current params don't exceed the abort criteria for the next step
        current_res_log2 = min(current_res_log2, highest_res_log2)
        current_target_time = min(current_target_time, max_target_time)
        
        # reset the flags
        convergence_maintained = false;
        lower_bound_only = true;
    end

end

main (generic function with 1 method)

In [31]:
main()

persistent random seed: 1960099
current mass: 0
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8950226767997562
		max |amplitude| chi before rescaling: 1.2285685267817748
  3.036475 seconds (818.66 k allocations: 1.102 GiB, 4.52% gc time, 61.11% compilation time: 63% of which was recompilation)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8955090461650976
		max |amplitude| chi before rescaling: 1.2286582135676491
  3.483225 seconds (861.71 k allocations: 4.029 GiB, 22.18% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8955090461650976
		max |amplitude| chi before rescaling: 1.2286582135676491
  7.096645 seconds (2.35 M allocations: 15.504 GiB, 16.84% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence kept at all times.
No runaway detected.
Finished plotting.
Saved

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/00_THANOS_TEMP/L=1_C4=1_m2=1/03_mass/03_rand/plots/1960099/0/animation_Nx=2048.gif


Terminating because one of the fields grew too large at time t = 3.398046874999049.
  3.583474 seconds (1.43 M allocations: 3.436 GiB, 8.98% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8955090461650976
		max |amplitude| chi before rescaling: 1.2286582135676491
Terminating because one of the fields grew too large at time t = 3.544726562497846.
  7.614773 seconds (2.94 M allocations: 13.810 GiB, 11.63% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8955090461650976
		max |amplitude| chi before rescaling: 1.2286582135676491
Terminating because one of the fields grew too large at time t = 3.6008300781295395.
 27.806376 seconds (8.31 M allocations: 54.881 GiB, 13.67% gc time)
... terminated
Output plot directory already exists.
Convergence lost at time t=2.356
Runaway detected at time t=1.384
Finished plotting.
Saved data.
Target time reset to 

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/00_THANOS_TEMP/L=1_C4=1_m2=1/03_mass/03_rand/plots/1960099/0/animation_Nx=2048.gif


persistent random seed: 1960099
current mass: 16
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8950226767997562
		max |amplitude| chi before rescaling: 1.2285685267817748
Terminating because one of the fields grew too large at time t = 3.1537109374992713.
  3.180287 seconds (1.34 M allocations: 3.194 GiB, 24.71% gc time, 1.72% compilation time: 100% of which was recompilation)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8955090461650976
		max |amplitude| chi before rescaling: 1.2286582135676491
Terminating because one of the fields grew too large at time t = 3.154492187498201.
  6.431824 seconds (2.62 M allocations: 12.298 GiB, 9.41% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8955090461650976
		max |amplitude| chi before rescaling: 1.2286582135676491
Terminating because one of the fields grew too

...

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/00_THANOS_TEMP/L=1_C4=1_m2=1/03_mass/03_rand/plots/1960099/224/animation_Nx=2048.gif


Terminating because one of the fields grew too large at time t = 8.563671874996913.
  5.286205 seconds (3.53 M allocations: 8.516 GiB, 10.75% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8955090461650976
		max |amplitude| chi before rescaling: 1.2286582135676491
Terminating because one of the fields grew too large at time t = 8.599804687514165.
 24.192484 seconds (7.07 M allocations: 33.225 GiB, 9.57% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8955090461650976
		max |amplitude| chi before rescaling: 1.2286582135676491
Terminating because one of the fields grew too large at time t = 8.600292968761808.
 84.418724 seconds (19.76 M allocations: 130.532 GiB, 8.37% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence lost at time t=7.8177709775963695
Runaway detected at time t=5.767449909887133
Finished pl

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/00_THANOS_TEMP/L=1_C4=1_m2=1/03_mass/03_rand/plots/1960099/240/animation_Nx=2048.gif


Terminating because one of the fields grew too large at time t = 9.015429687498557.
  6.758356 seconds (3.72 M allocations: 8.962 GiB, 9.71% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8955090461650976
		max |amplitude| chi before rescaling: 1.2286582135676491
Terminating because one of the fields grew too large at time t = 9.055566406265823.
 20.745161 seconds (7.44 M allocations: 34.979 GiB, 9.83% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.8955090461650976
		max |amplitude| chi before rescaling: 1.2286582135676491
Terminating because one of the fields grew too large at time t = 9.055468750005184.
 94.670257 seconds (20.80 M allocations: 137.428 GiB, 8.41% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence lost at time t=8.05826290330929
Runaway detected at time t=6.145601280344828
Finished plott

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/00_THANOS_TEMP/L=1_C4=1_m2=1/03_mass/03_rand/plots/1960099/256/animation_Nx=2048.gif


### export .jl for production run

In [2]:
using NBInclude
nbexport("main.jl", "main.ipynb")